# 03 — Two-Stage Classification

Ide: memecah masalah 35 kelas menjadi dua tahap yang lebih sederhana.
Dua varian dievaluasi di notebook ini:

**Varian A — binary (PASS vs kontrak):**
- **Stage 1** (binary): apakah board berakhir **PASS** atau **kontrak**?
- **Stage 2** (34 kelas): jika non-PASS, klasifikasikan level+strain kontrak
  di antara 34 kontrak yang mungkin.

**Varian B — kaskade kategori** (lihat bagian 6-9): stage 1 langsung
menentukan **kategori** (Pass/Partscore/Game/SmallSlam/GrandSlam, 5 kelas,
memakai `target_category` yang sudah ada di dataset), baru stage 2
mencari level+strain **persis** di dalam kategori yang terpilih — 4
classifier terpisah, satu per kategori non-Pass.

Untuk kedua varian, dua metode kombinasi dievaluasi:
1. **Hard routing** — stage 1 menentukan kategori/PASS final secara keras;
   stage 2 hanya dipakai untuk baris sesuai prediksi stage 1.
2. **Probability chaining** — `P(kelas) = P(kategori | stage1) x P(kelas | stage2)`
   digabung jadi satu distribusi probabilitas 35 kelas, supaya top-k
   accuracy tetap bisa dihitung dan stage 2 bisa "mengoreksi" keputusan
   stage 1 yang tipis di margin.

Dibandingkan dengan single-stage XGBoost baseline (model terbaik saat ini).
Ringkasan: **kaskade kategori (Varian B) mengungguli binary (Varian A)** —
lihat bagian 10 untuk detail.


In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-09" / "outputs" / "two_stage_classification"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed")

X_train = df_train[feature_cols].values
X_val = df_val[feature_cols].values
y_train = df_train["label"].values
y_val = df_val["label"].values

PASS_LABEL = "PASS"
pass_idx = list(le.classes_).index(PASS_LABEL)
print(f"PASS index in original label encoder: {pass_idx}")
print(f"Train PASS ratio: {(df_train['target_base'] == PASS_LABEL).mean():.3f}")


PASS index in original label encoder: 34
Train PASS ratio: 0.002


## 1. Baseline single-stage XGBoost (default `configs/config.yaml`)

In [2]:
BASE_XGB_PARAMS = dict(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)

t0 = time.time()
baseline_clf = XGBClassifier(**BASE_XGB_PARAMS)
baseline_clf.fit(X_train, y_train)
y_pred_baseline = baseline_clf.predict(X_val)
y_proba_baseline = baseline_clf.predict_proba(X_val)
res_baseline = evaluate(y_val, y_pred_baseline, y_proba_baseline, le, model_name="XGBoost (single-stage)")
print(f"Baseline fit: {time.time() - t0:.1f}s")
print_summary(res_baseline)


Baseline fit: 29.6s

  XGBoost (single-stage)
  Accuracy          : 0.5127
  Precision (macro) : 0.3335
  Recall (macro)    : 0.2515
  F1 (macro)        : 0.2685
  F1 (weighted)     : 0.4685
  Top-3 Accuracy    : 0.7508
  Top-5 Accuracy    : 0.8480


## 2. Stage 1 — PASS vs non-PASS (binary)

In [3]:
y_train_bin = (df_train["target_base"] != PASS_LABEL).astype(int).values
y_val_bin = (df_val["target_base"] != PASS_LABEL).astype(int).values

stage1_clf = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, eval_metric="logloss", verbosity=0,
)
t0 = time.time()
stage1_clf.fit(X_train, y_train_bin)
print(f"Stage 1 fit: {time.time() - t0:.1f}s")

y_val_bin_pred = stage1_clf.predict(X_val)
y_val_bin_proba = stage1_clf.predict_proba(X_val)  # column 1 = P(non-PASS)

stage1_metrics = {
    "accuracy": float(accuracy_score(y_val_bin, y_val_bin_pred)),
    "precision": float(precision_score(y_val_bin, y_val_bin_pred)),
    "recall": float(recall_score(y_val_bin, y_val_bin_pred)),
    "f1": float(f1_score(y_val_bin, y_val_bin_pred)),
}
print("Stage 1 (is-contract) metrics on val:", stage1_metrics)


Stage 1 fit: 0.4s
Stage 1 (is-contract) metrics on val: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


## 3. Stage 2 — kontrak spesifik (34 kelas, hanya baris non-PASS)

In [4]:
train_nonpass = df_train[df_train["target_base"] != PASS_LABEL]
val_nonpass = df_val[df_val["target_base"] != PASS_LABEL]

le2 = LabelEncoder()
y_train_nonpass = le2.fit_transform(train_nonpass["target_base"])
X_train_nonpass = train_nonpass[feature_cols].values

X_val_nonpass = val_nonpass[feature_cols].values
y_val_nonpass = le2.transform(val_nonpass["target_base"])

print(f"Stage 2 classes: {len(le2.classes_)} (should be 34 = 35 - PASS)")
print(f"Stage 2 train rows: {len(train_nonpass)}, val rows: {len(val_nonpass)}")

stage2_clf = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)
t0 = time.time()
stage2_clf.fit(X_train_nonpass, y_train_nonpass)
print(f"Stage 2 fit: {time.time() - t0:.1f}s")

y_val_nonpass_pred = stage2_clf.predict(X_val_nonpass)
stage2_acc_on_true_nonpass = accuracy_score(y_val_nonpass, y_val_nonpass_pred)
print(f"Stage 2 accuracy (conditional on TRUE non-PASS rows): {stage2_acc_on_true_nonpass:.4f}")


Stage 2 classes: 34 (should be 34 = 35 - PASS)
Stage 2 train rows: 7143, val rows: 1529


Stage 2 fit: 29.8s
Stage 2 accuracy (conditional on TRUE non-PASS rows): 0.5114


## 4. Gabungkan stage 1 + stage 2

Map indeks kelas stage 2 (`le2`, 34 kelas) balik ke indeks kelas original
(`le`, 35 kelas termasuk PASS) supaya bisa dibandingkan langsung dengan
baseline single-stage lewat fungsi `evaluate()` yang sama.


In [5]:
stage2_to_orig_idx = np.array([list(le.classes_).index(c) for c in le2.classes_])
n_classes = len(le.classes_)

stage2_proba_val = stage2_clf.predict_proba(X_val)  # full val set, shape (n, 34)

# --- Method A: hard routing ---
y_pred_hard = np.full(len(X_val), pass_idx, dtype=int)
nonpass_mask = y_val_bin_pred == 1
stage2_hard_pred_orig = stage2_to_orig_idx[stage2_clf.predict(X_val[nonpass_mask])]
y_pred_hard[nonpass_mask] = stage2_hard_pred_orig

# --- Method B: probability chaining ---
p_nonpass = y_val_bin_proba[:, 1]
p_pass = y_val_bin_proba[:, 0]
combined_proba = np.zeros((len(X_val), n_classes))
combined_proba[:, pass_idx] = p_pass
combined_proba[:, stage2_to_orig_idx] += p_nonpass[:, None] * stage2_proba_val
y_pred_soft = combined_proba.argmax(axis=1)

res_hard = evaluate(y_val, y_pred_hard, combined_proba, le, model_name="Two-stage (hard routing)")
res_soft = evaluate(y_val, y_pred_soft, combined_proba, le, model_name="Two-stage (proba chaining)")

print_summary(res_hard)
print_summary(res_soft)



  Two-stage (hard routing)
  Accuracy          : 0.5127
  Precision (macro) : 0.3324
  Recall (macro)    : 0.2513
  F1 (macro)        : 0.2685
  F1 (weighted)     : 0.4687
  Top-3 Accuracy    : 0.7632
  Top-5 Accuracy    : 0.8461

  Two-stage (proba chaining)
  Accuracy          : 0.5127
  Precision (macro) : 0.3324
  Recall (macro)    : 0.2513
  F1 (macro)        : 0.2685
  F1 (weighted)     : 0.4687
  Top-3 Accuracy    : 0.7632
  Top-5 Accuracy    : 0.8461


## 5. Bandingkan dengan baseline single-stage

In [6]:
comparison = compare_models([res_baseline, res_hard, res_soft])
comparison


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
XGBoost (single-stage),0.51272,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010
Two-stage (hard routing),0.51272,0.332414,0.251274,0.268499,0.468666,0.763209,0.846053
Two-stage (proba chaining),0.51272,0.332414,0.251274,0.268499,0.468666,0.763209,0.846053


## 6. Pendekatan alternatif — kaskade kategori

Permintaan lanjutan: daripada stage 1 cuma PASS vs non-PASS, coba stage 1
langsung menentukan **kategori** kontrak dulu — `target_category` yang
sudah ada di dataset (5 kelas: Pass, Partscore, Game, SmallSlam,
GrandSlam) — baru stage 2 mencari **level+strain persis** di dalam
kategori yang terpilih.

Distribusi kategori di train (jauh lebih seimbang antar kategori besar
dibanding PASS vs non-PASS yang timpang ekstrem 15 vs 7.141):

| Kategori | n (train) | Kelas kontrak di dalamnya |
|---|---|---|
| Pass | 15 | 1 (PASS saja) |
| Partscore | 2.769 | 19 (1C..5S non-game) |
| Game | 3.858 | 5 (3N, 4H, 4S, 5C, 5D) |
| SmallSlam | 465 | 5 (6C..6N) |
| GrandSlam | 49 | 5 (7C..7N) |

Stage 2 di sini berarti **4 classifier terpisah** (satu per kategori
non-Pass), masing-masing hanya melihat baris dari kategorinya sendiri.
GrandSlam classifier dilatih di atas 49 baris saja — hasil untuk kategori
ini diperkirakan lemah, tapi tetap dilaporkan apa adanya.


In [7]:
from sklearn.utils.class_weight import compute_sample_weight

cat_le = LabelEncoder()
y_train_cat = cat_le.fit_transform(df_train["target_category"])
y_val_cat = cat_le.transform(df_val["target_category"])
print("Kategori:", list(cat_le.classes_))
print(df_train["target_category"].value_counts())

stage1_cat_clf = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)
t0 = time.time()
sw_cat = compute_sample_weight("balanced", y_train_cat)
stage1_cat_clf.fit(X_train, y_train_cat, sample_weight=sw_cat)
print(f"Stage 1 (kategori) fit: {time.time() - t0:.1f}s")

stage1_cat_pred = stage1_cat_clf.predict(X_val)
stage1_cat_proba = stage1_cat_clf.predict_proba(X_val)

res_stage1_cat = evaluate(y_val_cat, stage1_cat_pred, stage1_cat_proba,
                           cat_le, model_name="Stage 1 (kategori, 5 kelas)")
print_summary(res_stage1_cat)
print()
print("Per-kategori precision/recall/F1:")
for cat_name, row in res_stage1_cat["per_class_report"].items():
    if cat_name in cat_le.classes_:
        precision = row["precision"]
        recall = row["recall"]
        f1 = row["f1-score"]
        support = row["support"]
        print(f"  {cat_name:12s} precision={precision:.3f} recall={recall:.3f} f1={f1:.3f} support={support:.0f}")


Kategori: ['Game', 'GrandSlam', 'Partscore', 'Pass', 'SmallSlam']
target_category
Game         3861
Partscore    2767
SmallSlam     469
GrandSlam      46
Pass           14
Name: count, dtype: int64


Stage 1 (kategori) fit: 4.3s

  Stage 1 (kategori, 5 kelas)
  Accuracy          : 0.7573
  Precision (macro) : 0.6142
  Recall (macro)    : 0.6340
  F1 (macro)        : 0.6230
  F1 (weighted)     : 0.7557
  Top-3 Accuracy    : 0.9954

Per-kategori precision/recall/F1:
  Game         precision=0.807 recall=0.760 f1=0.783 support=825
  GrandSlam    precision=0.000 recall=0.000 f1=0.000 support=13
  Partscore    precision=0.738 recall=0.791 f1=0.764 support=594
  Pass         precision=1.000 recall=1.000 f1=1.000 support=4
  SmallSlam    precision=0.526 recall=0.619 f1=0.569 support=97


## 7. Stage 2 — kontrak persis di dalam tiap kategori

In [8]:
NONPASS_CATEGORIES = [c for c in cat_le.classes_ if c != "Pass"]

stage2_cat_models = {}
stage2_cat_cond_acc = {}
for cat_name in NONPASS_CATEGORIES:
    sub_train = df_train[df_train["target_category"] == cat_name]
    sub_val = df_val[df_val["target_category"] == cat_name]

    local_le = LabelEncoder()
    y_sub_train = local_le.fit_transform(sub_train["target_base"])

    clf = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, eval_metric="mlogloss", verbosity=0,
    )
    t0 = time.time()
    clf.fit(sub_train[feature_cols].values, y_sub_train)

    if len(sub_val) > 0:
        y_sub_val = local_le.transform(sub_val["target_base"])
        y_sub_val_pred = clf.predict(sub_val[feature_cols].values)
        cond_acc = accuracy_score(y_sub_val, y_sub_val_pred)
    else:
        cond_acc = float("nan")

    stage2_cat_models[cat_name] = (clf, local_le)
    stage2_cat_cond_acc[cat_name] = cond_acc
    print(f"{cat_name:12s} train_n={len(sub_train):5d} val_n={len(sub_val):4d} "
          f"classes={len(local_le.classes_)} "
          f"conditional_acc={cond_acc:.4f} ({time.time() - t0:.1f}s)")


Game         train_n= 3861 val_n= 825 classes=5 conditional_acc=0.7830 (3.8s)


GrandSlam    train_n=   46 val_n=  13 classes=5 conditional_acc=0.1538 (0.4s)


Partscore    train_n= 2767 val_n= 594 classes=19 conditional_acc=0.4327 (14.2s)


SmallSlam    train_n=  469 val_n=  97 classes=5 conditional_acc=0.7732 (1.4s)


## 8. Gabungkan kaskade kategori (hard routing & probability chaining)

Sama seperti pendekatan binary di atas: **hard routing** memakai kategori
hasil prediksi stage 1 apa adanya, **probability chaining** mengalikan
`P(kategori) x P(kontrak | kategori)` supaya tetap dapat distribusi 35
kelas penuh untuk top-k accuracy.


In [9]:
cat_name_to_idx = {name: i for i, name in enumerate(cat_le.classes_)}
pass_cat_idx = cat_name_to_idx["Pass"]

# --- Hard routing ---
y_pred_cat_hard = np.full(len(X_val), pass_idx, dtype=int)
for cat_name in NONPASS_CATEGORIES:
    clf, local_le = stage2_cat_models[cat_name]
    mask = stage1_cat_pred == cat_name_to_idx[cat_name]
    if mask.sum() == 0:
        continue
    local_pred = clf.predict(X_val[mask])
    local_labels = local_le.inverse_transform(local_pred)
    y_pred_cat_hard[mask] = [list(le.classes_).index(lbl) for lbl in local_labels]

# --- Probability chaining ---
combined_proba_cat = np.zeros((len(X_val), n_classes))
combined_proba_cat[:, pass_idx] = stage1_cat_proba[:, pass_cat_idx]
for cat_name in NONPASS_CATEGORIES:
    clf, local_le = stage2_cat_models[cat_name]
    p_cat = stage1_cat_proba[:, cat_name_to_idx[cat_name]]
    local_proba = clf.predict_proba(X_val)  # (n, n_local_classes)
    local_to_orig_idx = np.array([list(le.classes_).index(c) for c in local_le.classes_])
    combined_proba_cat[:, local_to_orig_idx] += p_cat[:, None] * local_proba

y_pred_cat_soft = combined_proba_cat.argmax(axis=1)

res_cat_hard = evaluate(y_val, y_pred_cat_hard, combined_proba_cat, le,
                         model_name="Two-stage kaskade-kategori (hard routing)")
res_cat_soft = evaluate(y_val, y_pred_cat_soft, combined_proba_cat, le,
                         model_name="Two-stage kaskade-kategori (proba chaining)")

print_summary(res_cat_hard)
print_summary(res_cat_soft)



  Two-stage kaskade-kategori (hard routing)
  Accuracy          : 0.4912
  Precision (macro) : 0.3105
  Recall (macro)    : 0.2698
  F1 (macro)        : 0.2690
  F1 (weighted)     : 0.4829
  Top-3 Accuracy    : 0.7626
  Top-5 Accuracy    : 0.8565

  Two-stage kaskade-kategori (proba chaining)
  Accuracy          : 0.5127
  Precision (macro) : 0.3197
  Recall (macro)    : 0.2655
  F1 (macro)        : 0.2677
  F1 (weighted)     : 0.4910
  Top-3 Accuracy    : 0.7626
  Top-5 Accuracy    : 0.8565


## 9. Bandingkan SEMUA pendekatan

In [10]:
all_results = [res_baseline, res_hard, res_soft, res_cat_hard, res_cat_soft]
comparison_all = compare_models(all_results)
comparison_all.to_csv(OUT_DIR / "val_comparison.csv")

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "stage1_binary_metrics": stage1_metrics,
    "stage2_accuracy_conditional_on_true_nonpass": float(stage2_acc_on_true_nonpass),
    "stage1_category_accuracy": float(res_stage1_cat["accuracy"]),
    "stage2_category_conditional_accuracy": stage2_cat_cond_acc,
    "results": {r["model"]: {k: r[k] for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]}
        for r in all_results},
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved: {OUT_DIR / 'val_comparison.csv'}")
print(f"Saved: {OUT_DIR / 'summary.json'}")

comparison_all


Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\two_stage_classification\val_comparison.csv
Saved: D:\Bridge-Prediction\experiments\2026-07-09\outputs\two_stage_classification\summary.json


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
XGBoost (single-stage),0.512720,0.333505,0.251472,0.268524,0.468546,0.750815,0.848010
Two-stage (hard routing),0.512720,0.332414,0.251274,0.268499,0.468666,0.763209,0.846053
Two-stage (proba chaining),0.512720,0.332414,0.251274,0.268499,0.468666,0.763209,0.846053
Two-stage kaskade-kategori (hard routing),0.491194,0.310546,0.269819,0.269030,0.482924,0.762557,0.856491
Two-stage kaskade-kategori (proba chaining),0.512720,0.319653,0.265475,0.267690,0.490952,0.762557,0.856491


## 10. Kesimpulan

**Diperbarui 2026-07-10 setelah perbaikan split group-aware** — angka di
bawah ini MENGGANTIKAN kesimpulan sebelumnya.

Hasil val set — semua pendekatan (`experiments/2026-07-09/outputs/two_stage_classification/val_comparison.csv`):

| Model | Accuracy | F1 macro | F1 weighted | Top-3 | Top-5 |
|---|---|---|---|---|---|
| XGBoost (single-stage) | 51.3% | 0.269 | 0.469 | 75.1% | 84.8% |
| Two-stage binary (hard/soft — identik) | 51.3% | 0.268 | 0.469 | 76.3% | 84.6% |
| Two-stage kaskade-kategori (hard routing) | 49.1% | 0.269 | 0.483 | 76.3% | **85.6%** |
| **Two-stage kaskade-kategori (proba chaining)** | 51.3% | 0.268 | **0.491** | 76.3% | **85.6%** |

- **Keunggulan kaskade kategori menyusut banyak** dibanding klaim
  sebelumnya (dulu F1 weighted 0.508 vs baseline 0.491 — beda 0.017;
  sekarang 0.491 vs 0.469 — beda 0.022, arahnya sama tapi skalanya beda
  dan baseline-nya sendiri berubah). Top-3 accuracy yang dulu melonjak
  signifikan (77.9% vs 76.3%) SEKARANG HAMPIR SAMA (76.3% vs 76.3%,
  tidak ada beda) — sebagian besar "kemenangan top-3/top-5" versi lama
  ternyata berasal dari kebocoran split, bukan struktur kaskade itu
  sendiri. Top-5 tetap unggul tipis (85.6% vs 84.8%).
- **Two-stage binary (PASS vs non-PASS) sekarang praktis TIDAK
  memberi perbaikan** dibanding single-stage (F1 weighted 0.469 vs
  0.469, identik) — berbeda dari versi sebelumnya yang menunjukkan
  perbaikan tipis (0.497 vs 0.491). Stage 1 tetap sempurna
  (`opening_strain_PASS` masih proxy 100% akurat untuk PASS, itu bukan
  soal split), tapi manfaatnya untuk skor keseluruhan hilang setelah
  split diperbaiki.
- **Diagnostik stage 2 juga berubah**: Partscore 43.3% (mirip),
  Game 78.3% (turun dari 80.1%), SmallSlam 77.3% (naik dari 76.5%),
  **GrandSlam anjlok ke 15.4%** (dari 36.4%) — makin menegaskan
  GrandSlam terlalu sedikit data (train sekarang bahkan lebih sedikit
  untuk kategori ini karena grouping) untuk classifier terpisah yang
  andal.
- **Kesimpulan yang tetap bertahan**: kaskade kategori masih lebih baik
  dari kaskade binary, dan keduanya masih lebih baik dari baseline di
  F1 weighted — arah temuan tidak berubah, hanya besarannya yang jauh
  lebih moderat dari klaim awal.
- **Rekomendasi diperbarui**: kaskade kategori tetap kandidat yang valid
  tapi keunggulannya lebih tipis dari yang dilaporkan sebelumnya —
  validasi di test set jadi lebih penting dari sebelumnya untuk
  memastikan perbaikan ini nyata, bukan noise sampling val set (~1.500
  baris).
